In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.insert(0, "../")

from data.features import (
    agg_crops,
    agg_surplus,
    agg_weather,
    agg_weather_w_lag,
    daily_nitrate,
    lagged_sensor_nitrate,
    nitrate_rolling,
    nitrate_avg_seasonal,
    nitrate_avg_calendar,
    doy_climatology_pure_signal,
)
from data.transforms import flatten_buckets, merge_on_date, match_seasonal
from data import get_site_ids

## Question: Is lagged weather data helpful?

In [5]:
from cook import *
from recipes2 import _covariates


def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (agg_weather(site_uid, edges=[])
                   .sort_values("date").set_index("date").asfreq("D")  # regular daily index
                   .shift(lag))                                        # actually lag by `lag` days
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]       # suffix the value columns
            return wdf.reset_index()
        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        out = merge_on_date([n_daily, *parts], spine=n_daily.index)
        return out.dropna(subset=["nitrate_con"]).reset_index(drop=True)
    return recipe

recipe = recipe_lagger([1])

lags = [1, 2, 3, 7, 10, 14, 21, 30]
recipes = {f"Lags {lags[:i]}" : recipe_lagger(lags[:i]) for i in range(len(lags))}
print(compare_many(recipes, **FAST_XGB))
    

compare_many: [5/8] Lags [1, 2, 3, 7]            elapsed 1927s

ValueError: no sites produced a usable frame (85 sites skipped) -- 85x MergeError: Passing 'suffixes' which cause duplicate columns {'vpd_lag1_x', 'fuel_moisture_1000h_lag1_x', 'min_rel_humidity_lag1_x', 'max_temp_lag1_y', 'vpd_lag1_y', 'max_rel_humidity_lag1_x', 'evapotranspiration_lag1_y', 'solar_rad_lag1_y', 'solar_rad_lag1_x', 'precip_in_1d_lag1_y', 'min_rel_humidity_lag1_y', 'evapotranspiration_lag1_x', 'min_temp_lag1_x', 'max_temp_lag1_x', 'precip_in_1d_lag1_x', 'min_temp_lag1_y', 'max_rel_humidity_lag1_y', 'fuel_moisture_1000h_lag1_y'} is not allowed.

In [ ]:
from cook import *
from recipes2 import _covariates

def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (agg_weather(site_uid, edges=[])
                   .sort_values("date").set_index("date").asfreq("D")  # regular daily index
                   .shift(lag))                                        # actually lag by `lag` days
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]       # suffix the value columns
            return wdf.reset_index()
        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        v = nitrate_violations(site_uid, threshold=10).rename("violation")
        out = merge_on_date([v, *parts], spine=n_daily.index)
        
        return out.dropna(subset=["nitrate_con"]).reset_index(drop=True)
    return recipe

recipe = recipe_lagger([3])
compare_many([recipe], task="clf", **FAST_XGB)
"""
lags = [1, 2, 3, 7, 10, 14, 21, 30]
recipes = {f"Lags {lags[:i]}" : recipe_lagger(lags[:i]) for i in range(len(lags))}
print(compare_many(recipes, task="clf", **FAST_XGB))
"""

compare_many: [1/8] Lags []                      elapsed    0s

KeyboardInterrupt: 